In [1]:
#imports and paths
import duckdb
import pandas as pd
import numpy as np
import holidays
import os

con = duckdb.connect()
clean_path = "/home/iai-labor/traffic_project/traffic_hourly_clean_v3.parquet"
features_path = "/home/iai-labor/traffic_project/traffic_features_v1.parquet"

In [2]:
#schema + date range check (run this first, paste output here)
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{clean_path}')").df()

,column_name,column_type,null,key,default,extra
0,SENSOR_ID,BIGINT,YES,None,None,None
1,DATETIME,TIMESTAMP,YES,None,None,None
2,TRAFFIC_INTENSITY,DOUBLE,YES,None,None,None
3,ELEMENT_TYPE,VARCHAR,YES,None,None,None
4,OCCUPANCY,DOUBLE,YES,None,None,None
5,LOAD,DOUBLE,YES,None,None,None
6,AVERAGE_SPEED,DOUBLE,YES,None,None,None
7,ERROR_FLAG,VARCHAR,YES,None,None,None
8,LONGITUDE,DOUBLE,YES,None,None,None
9,LATITUDE,DOUBLE,YES,None,None,None


In [3]:
#date range + subclass confirmation
con.sql(f"""
    SELECT 
        MIN(DATETIME) AS start, MAX(DATETIME) AS end,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT SENSOR_ID) AS n_sensors
    FROM read_parquet('{clean_path}')
""").show()

con.sql(f"""
    SELECT ROAD_SUBCLASS, COUNT(*) AS n_rows, COUNT(DISTINCT SENSOR_ID) AS n_sensors
    FROM read_parquet('{clean_path}')
    GROUP BY ROAD_SUBCLASS
""").show()

┌─────────────────────┬─────────────────────┬───────────┬───────────┐
│        start        │         end         │  n_rows   │ n_sensors │
│      timestamp      │      timestamp      │   int64   │   int64   │
├─────────────────────┼─────────────────────┼───────────┼───────────┤
│ 2023-01-01 00:00:00 │ 2026-05-31 23:00:00 │ 139987008 │      4886 │
└─────────────────────┴─────────────────────┴───────────┴───────────┘

┌───────────────┬───────────┬───────────┐
│ ROAD_SUBCLASS │  n_rows   │ n_sensors │
│    varchar    │   int64   │   int64   │
├───────────────┼───────────┼───────────┤
│ URB_normal    │ 123496359 │      4295 │
│ URB_ramp      │   4168569 │       145 │
│ C30           │   3674931 │       143 │
│ M30           │   8647149 │       304 │
└───────────────┴───────────┴───────────┘



In [4]:
#Calendar features + holiday flag
import holidays as hols

# Build Spain/Madrid holiday list for full date range
madrid_holidays = hols.Spain(subdiv="MD", years=range(2023, 2027))
holiday_dates = sorted([str(d) for d in madrid_holidays.keys()])
holiday_list_sql = ", ".join([f"DATE '{d}'" for d in holiday_dates])

print(f"Total Madrid holidays 2023-2026: {len(holiday_dates)}")
print("Sample:", holiday_dates[:5])

Total Madrid holidays 2023-2026: 48
Sample: ['2023-01-06', '2023-03-20', '2023-04-06', '2023-04-07', '2023-05-01']


In [5]:
# Road subclass encoding lookup
# Models will learn the distinction; SHAP will still show per-class importance
subclass_encoding = {
    "URB_normal": 0,
    "URB_ramp":   1,
    "M30":        2,
    "C30":        3,
}
subclass_sql_case = " ".join([
    f"WHEN ROAD_SUBCLASS = '{k}' THEN {v}"
    for k, v in subclass_encoding.items()
])
print("Subclass encoding:", subclass_encoding)

Subclass encoding: {'URB_normal': 0, 'URB_ramp': 1, 'M30': 2, 'C30': 3}


In [6]:
# full feature set
# This is the core cell — runs entirely on disk via DuckDB, no pandas memory risk

feature_query = f"""
COPY (
    WITH base AS (
        SELECT
            SENSOR_ID,
            DATETIME,
            TRAFFIC_INTENSITY_FINAL AS target,

            -- ── Temporal features ──────────────────────────────────────────
            EXTRACT(hour   FROM DATETIME)::INTEGER AS hour,
            EXTRACT(dow    FROM DATETIME)::INTEGER AS dow,        -- 0=Sun, 6=Sat
            EXTRACT(month  FROM DATETIME)::INTEGER AS month,
            EXTRACT(year   FROM DATETIME)::INTEGER AS year,

            -- Season (meteorological, Northern Hemisphere)
            CASE
                WHEN EXTRACT(month FROM DATETIME) IN (12,1,2)  THEN 0  -- Winter
                WHEN EXTRACT(month FROM DATETIME) IN (3,4,5)   THEN 1  -- Spring
                WHEN EXTRACT(month FROM DATETIME) IN (6,7,8)   THEN 2  -- Summer
                WHEN EXTRACT(month FROM DATETIME) IN (9,10,11) THEN 3  -- Autumn
            END AS season,

            -- Cyclical encodings (hour and dow treated as circular)
            SIN(2 * PI() * EXTRACT(hour FROM DATETIME) / 24.0) AS hour_sin,
            COS(2 * PI() * EXTRACT(hour FROM DATETIME) / 24.0) AS hour_cos,
            SIN(2 * PI() * EXTRACT(dow  FROM DATETIME) / 7.0)  AS dow_sin,
            COS(2 * PI() * EXTRACT(dow  FROM DATETIME) / 7.0)  AS dow_cos,
            SIN(2 * PI() * EXTRACT(month FROM DATETIME) / 12.0) AS month_sin,
            COS(2 * PI() * EXTRACT(month FROM DATETIME) / 12.0) AS month_cos,

            -- Weekend flag (Saturday=6, Sunday=0 in DuckDB dow)
            CASE WHEN EXTRACT(dow FROM DATETIME) IN (0,6) THEN 1 ELSE 0 END AS is_weekend,

            -- Holiday flag
            CASE WHEN CAST(DATETIME AS DATE) IN ({holiday_list_sql})
                 THEN 1 ELSE 0 END AS is_holiday,

            -- Rush hour flag (morning 6-9, evening 17-20 as defined in thesis)
            CASE
                WHEN EXTRACT(hour FROM DATETIME) BETWEEN 6  AND 9  THEN 1
                WHEN EXTRACT(hour FROM DATETIME) BETWEEN 17 AND 20 THEN 1
                ELSE 0
            END AS is_rush_hour,

            -- ── Lag features (per sensor, temporal) ───────────────────────
            LAG(TRAFFIC_INTENSITY_FINAL, 1)
                OVER (PARTITION BY SENSOR_ID ORDER BY DATETIME) AS lag_1h,
            LAG(TRAFFIC_INTENSITY_FINAL, 2)
                OVER (PARTITION BY SENSOR_ID ORDER BY DATETIME) AS lag_2h,
            LAG(TRAFFIC_INTENSITY_FINAL, 3)
                OVER (PARTITION BY SENSOR_ID ORDER BY DATETIME) AS lag_3h,
            LAG(TRAFFIC_INTENSITY_FINAL, 24)
                OVER (PARTITION BY SENSOR_ID ORDER BY DATETIME) AS lag_24h,
            LAG(TRAFFIC_INTENSITY_FINAL, 168)
                OVER (PARTITION BY SENSOR_ID ORDER BY DATETIME) AS lag_168h,

            -- Rolling mean (last 3h, last 24h) as smoothed context
            AVG(TRAFFIC_INTENSITY_FINAL)
                OVER (PARTITION BY SENSOR_ID ORDER BY DATETIME
                      ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) AS roll_mean_3h,
            AVG(TRAFFIC_INTENSITY_FINAL)
                OVER (PARTITION BY SENSOR_ID ORDER BY DATETIME
                      ROWS BETWEEN 24 PRECEDING AND 1 PRECEDING) AS roll_mean_24h,

            -- ── Road static features ───────────────────────────────────────
            CASE {subclass_sql_case} END AS road_subclass_enc,
            LONGITUDE AS longitude,
            LATITUDE  AS latitude,

            -- ── Weather features ───────────────────────────────────────────
            TEMPERATURE,
            PRECIPITATION,
            WIND_SPEED,
            RELATIVE_HUMIDITY,
            SOLAR_RADIATION,
            BAROMETRIC_PRESSURE,
            WIND_DIRECTION,

            -- ── Provenance quality flags (useful for weighted loss later) ──
            IS_IMPUTED_V2::INTEGER    AS is_imputed,
            IS_SEASONAL_FILLED::INTEGER AS is_seasonal_filled,

            -- ── Split assignment ───────────────────────────────────────────
            CASE
                WHEN DATETIME < '2025-06-01' THEN 'train'
                WHEN DATETIME < '2025-12-01' THEN 'val'
                ELSE 'test'
            END AS split

            

        FROM read_parquet('{clean_path}')
    )
    -- Drop the first 168 rows per sensor (lag_168h will be NULL there)
    -- Keep them in the dataset but flag them — models will handle NaN lags
    SELECT * FROM base
)
TO '{features_path}'
(FORMAT PARQUET, COMPRESSION 'snappy')
"""

import time
t0 = time.time()
con.sql(feature_query)
print(f"Done in {time.time()-t0:.1f}s")

Done in 34.4s


In [7]:
#Verify output
con.sql(f"""
    SELECT split,
           COUNT(*) AS n_rows,
           COUNT(DISTINCT SENSOR_ID) AS n_sensors,
           MIN(DATETIME) AS start,
           MAX(DATETIME) AS end
    FROM read_parquet('{features_path}')
    GROUP BY split
    ORDER BY start
""").show()

┌─────────┬──────────┬───────────┬─────────────────────┬─────────────────────┐
│  split  │  n_rows  │ n_sensors │        start        │         end         │
│ varchar │  int64   │   int64   │      timestamp      │      timestamp      │
├─────────┼──────────┼───────────┼─────────────────────┼─────────────────────┤
│ train   │ 98150766 │      4799 │ 2023-01-01 00:00:00 │ 2025-05-31 23:00:00 │
│ val     │ 20948766 │      4783 │ 2025-06-01 00:00:00 │ 2025-11-30 23:00:00 │
│ test    │ 20887476 │      4842 │ 2025-12-01 00:00:00 │ 2026-05-31 23:00:00 │
└─────────┴──────────┴───────────┴─────────────────────┴─────────────────────┘



In [8]:
# Check lag NULLs
con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE lag_1h   IS NULL) AS null_lag_1h,
        COUNT(*) FILTER (WHERE lag_24h  IS NULL) AS null_lag_24h,
        COUNT(*) FILTER (WHERE lag_168h IS NULL) AS null_lag_168h,
        COUNT(*) AS total_rows
    FROM read_parquet('{features_path}')
""").show()

┌─────────────┬──────────────┬───────────────┬────────────┐
│ null_lag_1h │ null_lag_24h │ null_lag_168h │ total_rows │
│    int64    │    int64     │     int64     │   int64    │
├─────────────┼──────────────┼───────────────┼────────────┤
│      671620 │       783501 │       1482031 │  139987008 │
└─────────────┴──────────────┴───────────────┴────────────┘



In [9]:
#Check weather feature nulls
con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE TEMPERATURE     IS NULL) AS null_temp,
        COUNT(*) FILTER (WHERE PRECIPITATION   IS NULL) AS null_precip,
        COUNT(*) FILTER (WHERE WIND_SPEED      IS NULL) AS null_wind,
        COUNT(*) FILTER (WHERE RELATIVE_HUMIDITY IS NULL) AS null_humidity,
        COUNT(*) FILTER (WHERE SOLAR_RADIATION IS NULL) AS null_solar,
        COUNT(*) AS total
    FROM read_parquet('{features_path}')
""").show()

┌───────────┬─────────────┬───────────┬───────────────┬────────────┬───────────┐
│ null_temp │ null_precip │ null_wind │ null_humidity │ null_solar │   total   │
│   int64   │    int64    │   int64   │     int64     │   int64    │   int64   │
├───────────┼─────────────┼───────────┼───────────────┼────────────┼───────────┤
│  17496362 │    87887175 │  96268678 │      30672028 │  104329193 │ 139987008 │
└───────────┴─────────────┴───────────┴───────────────┴────────────┴───────────┘



In [10]:
#Quick feature summary stats per split (sanity check)
con.sql(f"""
    SELECT split,
           AVG(target)          AS avg_target,
           AVG(TEMPERATURE)     AS avg_temp,
           AVG(lag_1h)          AS avg_lag1h,
           AVG(is_holiday)      AS pct_holiday,
           AVG(is_weekend)      AS pct_weekend,
           AVG(is_rush_hour)    AS pct_rush_hour
    FROM read_parquet('{features_path}')
    GROUP BY split ORDER BY split
""").show()

┌─────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┐
│  split  │     avg_target     │      avg_temp      │     avg_lag1h      │     pct_holiday     │     pct_weekend     │    pct_rush_hour    │
│ varchar │       double       │       double       │       double       │       double        │       double        │       double        │
├─────────┼────────────────────┼────────────────────┼────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│ test    │ 331.97976465902127 │  10.26758403912426 │ 331.96508634170345 │ 0.04950138542349492 │ 0.28568918523233733 │ 0.33333376421354116 │
│ train   │ 350.68295646323025 │  15.63662682558005 │ 350.68539318515144 │ 0.03405526147396547 │ 0.28571487664192047 │ 0.33333502460897757 │
│ val     │  334.0620215437111 │ 21.830110984547492 │  334.0867325098392 │ 0.01639313742871537 │ 0.28962546051638555 │ 0.33333338106884197 │
└─────────┴──

In [11]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{features_path}')").df()

,column_name,column_type,null,key,default,extra
0,SENSOR_ID,BIGINT,YES,None,None,None
1,DATETIME,TIMESTAMP,YES,None,None,None
2,target,DOUBLE,YES,None,None,None
3,hour,INTEGER,YES,None,None,None
4,dow,INTEGER,YES,None,None,None
5,month,INTEGER,YES,None,None,None
6,year,INTEGER,YES,None,None,None
7,season,INTEGER,YES,None,None,None
8,hour_sin,DOUBLE,YES,None,None,None
9,hour_cos,DOUBLE,YES,None,None,None


In [12]:
#Weather NULLs by road subclass
con.sql(f"""
    SELECT 
        road_subclass_enc,
        CASE road_subclass_enc 
            WHEN 0 THEN 'URB_normal'
            WHEN 1 THEN 'URB_ramp'
            WHEN 2 THEN 'M30'
            WHEN 3 THEN 'C30'
        END AS road_subclass,
        COUNT(*) AS n_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE WIND_SPEED IS NULL) / COUNT(*), 1)
            AS pct_null_wind,
        ROUND(100.0 * COUNT(*) FILTER (WHERE SOLAR_RADIATION IS NULL) / COUNT(*), 1)
            AS pct_null_solar,
        ROUND(100.0 * COUNT(*) FILTER (WHERE PRECIPITATION IS NULL) / COUNT(*), 1)
            AS pct_null_precip,
        ROUND(100.0 * COUNT(*) FILTER (WHERE TEMPERATURE IS NULL) / COUNT(*), 1)
            AS pct_null_temp
    FROM read_parquet('{features_path}')
    GROUP BY road_subclass_enc
    ORDER BY road_subclass_enc
""").show()

┌───────────────────┬───────────────┬───────────┬───────────────┬────────────────┬─────────────────┬───────────────┐
│ road_subclass_enc │ road_subclass │  n_rows   │ pct_null_wind │ pct_null_solar │ pct_null_precip │ pct_null_temp │
│       int32       │    varchar    │   int64   │    double     │     double     │     double      │    double     │
├───────────────────┼───────────────┼───────────┼───────────────┼────────────────┼─────────────────┼───────────────┤
│                 0 │ URB_normal    │ 123496359 │          68.2 │           74.3 │            62.3 │          12.2 │
│                 1 │ URB_ramp      │   4168569 │          72.0 │           78.1 │            62.4 │          15.2 │
│                 2 │ M30           │   8647149 │          64.8 │           68.2 │            56.5 │          12.7 │
│                 3 │ C30           │   3674931 │          93.0 │           93.7 │            93.7 │          18.2 │
└───────────────────┴───────────────┴───────────┴───────────────

In [13]:
#Weather NULLs by year
con.sql(f"""
    SELECT 
        year,
        ROUND(100.0 * COUNT(*) FILTER (WHERE WIND_SPEED IS NULL) / COUNT(*), 1)
            AS pct_null_wind,
        ROUND(100.0 * COUNT(*) FILTER (WHERE SOLAR_RADIATION IS NULL) / COUNT(*), 1)
            AS pct_null_solar,
        ROUND(100.0 * COUNT(*) FILTER (WHERE PRECIPITATION IS NULL) / COUNT(*), 1)
            AS pct_null_precip,
        ROUND(100.0 * COUNT(*) FILTER (WHERE TEMPERATURE IS NULL) / COUNT(*), 1)
            AS pct_null_temp,
        COUNT(*) AS n_rows
    FROM read_parquet('{features_path}')
    GROUP BY year ORDER BY year
""").show()

┌───────┬───────────────┬────────────────┬─────────────────┬───────────────┬──────────┐
│ year  │ pct_null_wind │ pct_null_solar │ pct_null_precip │ pct_null_temp │  n_rows  │
│ int32 │    double     │     double     │     double      │    double     │  int64   │
├───────┼───────────────┼────────────────┼─────────────────┼───────────────┼──────────┤
│  2023 │          70.4 │           75.6 │            63.9 │          12.4 │ 39799706 │
│  2024 │          67.0 │           73.2 │            61.3 │           9.4 │ 41184329 │
│  2025 │          67.3 │           73.4 │            61.1 │          10.0 │ 41674439 │
│  2026 │          72.7 │           77.8 │            67.8 │          26.0 │ 17328534 │
└───────┴───────────────┴────────────────┴─────────────────┴───────────────┴──────────┘



In [14]:
# Are NULLs simultaneous (station offline) or independent?
con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE WIND_SPEED IS NULL
                         AND SOLAR_RADIATION IS NULL
                         AND PRECIPITATION IS NULL)   AS all_three_null,
        COUNT(*) FILTER (WHERE WIND_SPEED IS NULL
                         AND SOLAR_RADIATION IS NOT NULL) AS wind_only_null,
        COUNT(*) FILTER (WHERE SOLAR_RADIATION IS NULL
                         AND WIND_SPEED IS NOT NULL)      AS solar_only_null,
        COUNT(*) FILTER (WHERE PRECIPITATION IS NULL
                         AND WIND_SPEED IS NOT NULL)      AS precip_only_null,
        COUNT(*) AS total
    FROM read_parquet('{features_path}')
""").show()

┌────────────────┬────────────────┬─────────────────┬──────────────────┬───────────┐
│ all_three_null │ wind_only_null │ solar_only_null │ precip_only_null │   total   │
│     int64      │     int64      │      int64      │      int64       │   int64   │
├────────────────┼────────────────┼─────────────────┼──────────────────┼───────────┤
│       84870418 │         225097 │         8285612 │          3008032 │ 139987008 │
└────────────────┴────────────────┴─────────────────┴──────────────────┴───────────┘



In [18]:
import osmnx as ox
import geopandas as gpd

# Query all relevant POI types within Madrid municipality boundary
# Using OSM tags that correspond to traffic-generating locations

tags = {
    # Transport
    "railway": ["station", "halt"],
    "public_transport": ["station", "stop_area"],
    "amenity": ["bus_station", "ferry_terminal"],
    
    # Large crowd generators
    "leisure": ["stadium", "sports_centre"],
    "amenity": ["hospital", "university", "college"],
    "shop": ["mall", "supermarket"],
    "tourism": ["attraction", "museum"],
}

# Pull from OSM (Madrid boundary)
pois = ox.features_from_place("Madrid, Spain", tags=tags)

print(pois.shape)
print(pois.columns.tolist())
print(pois["amenity"].value_counts() if "amenity" in pois.columns else "no amenity col")

(2129, 357)
['geometry', 'covered', 'fare_area', 'name', 'operator', 'public_transport', 'railway', 'subway', 'wheelchair', 'colour', 'network', 'toilets:wheelchair', 'source', 'train', 'wikidata', 'wikipedia', 'layer', 'source:name', 'wheelchair:description', 'station', 'amenity', 'highway', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:unit', 'brand', 'brand:wikidata', 'brand:wikipedia', 'level', 'shop', 'old_name', 'url', 'bicycle', 'bus', 'education', 'air_conditioning', 'contact:website', 'network:wikidata', 'operator:wikidata', 'platforms', 'toilets:menstrual_products', 'website', 'opening_hours', 'payment:cash', 'payment:mastercard', 'payment:visa', 'leisure', 'sport', 'wikimedia_commons', 'organic', 'created_by', 'payment:credit_cards', 'payment:debit_cards', 'addr:suburb', 'description', 'email', 'fax', 'tourism', 'check_date', 'official_name', 'operator:wikipedia', 'check_date:wheelchair', 'name:ar', 'name:es', 'branch', 'fee', 'internet_access', 'int

In [19]:
# Check what actually came back across all relevant tag columns
print("railway values:")
print(pois["railway"].value_counts() if "railway" in pois.columns else "missing")

print("\npublic_transport values:")
print(pois["public_transport"].value_counts() if "public_transport" in pois.columns else "missing")

print("\nleisure values:")
print(pois["leisure"].value_counts() if "leisure" in pois.columns else "missing")

print("\nshop values:")
print(pois["shop"].value_counts() if "shop" in pois.columns else "missing")

print("\ntourism values:")
print(pois["tourism"].value_counts() if "tourism" in pois.columns else "missing")

# Also check geometry types — some POIs are polygons not points
print("\nGeometry types:")
print(pois.geom_type.value_counts())


railway values:
railway
station            255
subway_entrance      1
yard                 1
Name: count, dtype: int64

public_transport values:
public_transport
station     298
platform      1
Name: count, dtype: int64

leisure values:
leisure
sports_centre    369
stadium           28
garden             4
escape_game        1
park               1
Name: count, dtype: int64

shop values:
shop
supermarket    855
mall            49
Name: count, dtype: int64

tourism values:
tourism
attraction    210
museum         98
Name: count, dtype: int64

Geometry types:
Point           1331
Polygon          786
MultiPolygon       8
LineString         4
Name: count, dtype: int64


In [20]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point

# Step 1: Filter to meaningful categories only
poi_categories = {
    "station":    pois[pois["railway"] == "station"].copy(),
    "stadium":    pois[pois["leisure"] == "stadium"].copy(),
    "hospital":   pois[pois["amenity"] == "hospital"].copy(),
    "university": pois[pois["amenity"] == "university"].copy(),
    "mall":       pois[pois["shop"] == "mall"].copy(),
    "bus_station":pois[pois["amenity"] == "bus_station"].copy(),
}

# Fixed Step 2: project to metric CRS before centroid, then back to WGS84
for cat, gdf in poi_categories.items():
    poi_categories[cat] = (
        gdf[["name", "geometry"]]
        .copy()
        .set_crs("EPSG:4326")
        .to_crs("EPSG:25830")        # project to meters first
        .assign(geometry=lambda g: g["geometry"].centroid)  # centroid in meters
        .to_crs("EPSG:4326")         # back to WGS84 for consistency
    )
    print(f"{cat}: {len(poi_categories[cat])} POIs")

station: 255 POIs
stadium: 28 POIs
hospital: 75 POIs
university: 81 POIs
mall: 49 POIs
bus_station: 18 POIs


In [21]:
# Step 3: Load sensor locations (small table, safe in pandas)
sensor_locations = con.sql(f"""
    SELECT DISTINCT SENSOR_ID, latitude AS LATITUDE, longitude AS LONGITUDE
    FROM read_parquet('{features_path}')
""").df()

sensors_gdf = gpd.GeoDataFrame(
    sensor_locations,
    geometry=gpd.points_from_xy(
        sensor_locations["LONGITUDE"],
        sensor_locations["LATITUDE"]
    ),
    crs="EPSG:4326"
)

print(f"Total sensors: {len(sensors_gdf)}")
print(sensor_locations.head(3))

Total sensors: 4886
   SENSOR_ID   LATITUDE  LONGITUDE
0       3805  40.394422  -3.640029
1       6793  40.369923  -3.720139
2       4394  40.425581  -3.707572


In [22]:
# Check how many sensors have null geometry
print(f"Sensors with null geometry: {sensors_gdf.geometry.isna().sum()}")
print(f"Sensors with null latitude:  {sensor_locations['LATITUDE'].isna().sum()}")
print(f"Sensors with null longitude: {sensor_locations['LONGITUDE'].isna().sum()}")

# Drop sensors with null coordinates before projecting
sensors_gdf_clean = sensors_gdf[
    sensors_gdf.geometry.notna() & 
    ~sensors_gdf.geometry.is_empty
].reset_index(drop=True)

print(f"\nSensors before: {len(sensors_gdf)}")
print(f"Sensors after dropping null geometry: {len(sensors_gdf_clean)}")

# Re-project clean sensors
sensors_proj = sensors_gdf_clean.to_crs("EPSG:25830")
sensor_coords = np.array([[geom.y, geom.x] for geom in sensors_proj.geometry])
print(f"NaN in sensor coords after fix: {np.isnan(sensor_coords).sum()}")

Sensors with null geometry: 0
Sensors with null latitude:  45
Sensors with null longitude: 45

Sensors before: 4886
Sensors after dropping null geometry: 4886
NaN in sensor coords after fix: 90


In [23]:
# Debug exactly where the NaN is coming from
print("NaN check on sensor_coords:")
print(f"  Total values: {sensor_coords.size}")
print(f"  NaN count: {np.isnan(sensor_coords).sum()}")
print(f"  NaN rows: {np.where(np.isnan(sensor_coords).any(axis=1))[0][:10]}")

# Check those specific sensors
nan_rows = np.where(np.isnan(sensor_coords).any(axis=1))[0]
print(sensors_gdf_clean.iloc[nan_rows][["SENSOR_ID", "LATITUDE", "LONGITUDE"]].head(10))

NaN check on sensor_coords:
  Total values: 9772
  NaN count: 90
  NaN rows: [ 175  235  269  386  430  704  775  991 1222 1320]
      SENSOR_ID  LATITUDE  LONGITUDE
175       10398       NaN        NaN
235        4240       NaN        NaN
269        6707       NaN        NaN
386   170996558       NaN        NaN
430       10781       NaN        NaN
704        3904       NaN        NaN
775       10397       NaN        NaN
991       10851       NaN        NaN
1222       6711       NaN        NaN
1320       3791       NaN        NaN


In [24]:
# Completely different approach — use scipy directly on lat/lon
# No projection needed, no BallTree, handles NaN gracefully
from scipy.spatial.distance import cdist

def haversine_matrix(sensors_latlon, pois_latlon):
    """
    sensors_latlon: (n_sensors, 2) array of [lat, lon] in degrees
    pois_latlon:    (n_pois, 2) array of [lat, lon] in degrees
    Returns: (n_sensors,) array of km distances to nearest POI
    """
    R = 6371.0
    s = np.radians(sensors_latlon)
    p = np.radians(pois_latlon)

    # Pairwise haversine
    dlat = s[:, 0:1] - p[:, 0]          # (n_sensors, n_pois)
    dlon = s[:, 1:2] - p[:, 1]
    a = np.sin(dlat/2)**2 + \
        np.cos(s[:, 0:1]) * np.cos(p[:, 0]) * np.sin(dlon/2)**2
    dist_km = 2 * R * np.arcsin(np.sqrt(a))  # (n_sensors, n_pois)
    return dist_km.min(axis=1)               # nearest POI per sensor

# Build clean sensor lat/lon array — drop rows where either is NaN
sensor_latlon_df = sensors_gdf_clean[["SENSOR_ID", "LATITUDE", "LONGITUDE"]].dropna()
sensor_latlon = sensor_latlon_df[["LATITUDE", "LONGITUDE"]].values
print(f"Clean sensors for distance calc: {len(sensor_latlon_df)}")
print(f"NaN in sensor_latlon: {np.isnan(sensor_latlon).sum()}")

Clean sensors for distance calc: 4841
NaN in sensor_latlon: 0


In [25]:
# Compute distances for each POI category
results = sensor_latlon_df[["SENSOR_ID"]].copy().reset_index(drop=True)

for cat, poi_gdf in poi_categories.items():
    # Get centroid lat/lon from the already-processed poi_categories
    poi_latlon = np.array([
        [geom.y, geom.x]
        for geom in poi_gdf.geometry
        if geom is not None and not geom.is_empty
    ])
    
    # Drop any remaining NaN coords in POIs
    poi_latlon = poi_latlon[~np.isnan(poi_latlon).any(axis=1)]
    print(f"{cat}: {len(poi_latlon)} valid POI coordinates")
    
    dist_km = haversine_matrix(sensor_latlon, poi_latlon)
    results[f"dist_{cat}_km"] = dist_km
    print(f"  min={dist_km.min():.3f}km  "
          f"max={dist_km.max():.2f}km  "
          f"mean={dist_km.mean():.2f}km")

print("\nDone!")

station: 255 valid POI coordinates
  min=0.002km  max=3.83km  mean=0.45km
stadium: 28 valid POI coordinates
  min=0.065km  max=8.83km  mean=1.73km
hospital: 75 valid POI coordinates
  min=0.015km  max=6.13km  mean=1.15km
university: 81 valid POI coordinates
  min=0.014km  max=6.74km  mean=1.37km
mall: 49 valid POI coordinates
  min=0.015km  max=4.84km  mean=1.15km
bus_station: 18 valid POI coordinates
  min=0.027km  max=10.90km  mean=2.29km

Done!


In [26]:
# Add binary POI flag + save
poi_cols = [f"dist_{cat}_km" for cat in poi_categories]
results["near_major_poi"] = (results[poi_cols].min(axis=1) < 0.5).astype(int)

print(f"Sensors within 500m of any major POI: "
      f"{results['near_major_poi'].sum()} of {len(results)}")

results.to_parquet(
    "/home/iai-labor/traffic_project/sensor_poi_features.parquet",
    index=False
)
print("Saved sensor_poi_features.parquet")

# Sensor 7023 check
s7023 = results[results["SENSOR_ID"] == 7023]
print("\nSensor 7023 POI distances:")
print(s7023[poi_cols + ["near_major_poi"]].T.to_string())

Sensors within 500m of any major POI: 3944 of 4841
Saved sensor_poi_features.parquet

Sensor 7023 POI distances:
                         4633
dist_station_km      0.449376
dist_stadium_km      2.123610
dist_hospital_km     1.700867
dist_university_km   0.660150
dist_mall_km         1.703572
dist_bus_station_km  1.025714
near_major_poi       1.000000


In [27]:
# Join POI features into the main features file
# DuckDB join — no pandas memory risk
con.sql(f"""
    COPY (
        SELECT f.*,
               p.dist_station_km,
               p.dist_stadium_km,
               p.dist_hospital_km,
               p.dist_university_km,
               p.dist_mall_km,
               p.dist_bus_station_km,
               p.near_major_poi
        FROM read_parquet('{features_path}') f
        LEFT JOIN read_parquet(
            '/home/iai-labor/traffic_project/sensor_poi_features.parquet'
        ) p ON f.SENSOR_ID = p.SENSOR_ID
    ) TO '/home/iai-labor/traffic_project/traffic_features_v1.parquet'
    (FORMAT PARQUET, COMPRESSION 'snappy')
""")

# Verify
con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(dist_station_km) AS n_with_poi,
           COUNT(*) - COUNT(dist_station_km) AS n_missing_poi
    FROM read_parquet('/home/iai-labor/traffic_project/traffic_features_v1.parquet')
""").show()

┌───────────┬────────────┬───────────────┐
│  n_rows   │ n_with_poi │ n_missing_poi │
│   int64   │   int64    │     int64     │
├───────────┼────────────┼───────────────┤
│ 139987008 │  139461647 │        525361 │
└───────────┴────────────┴───────────────┘



In [28]:
# Update your feature column list — add POI features
feature_cols = [
    # Temporal cyclical
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    # Calendar flags
    "is_weekend", "is_holiday", "is_rush_hour", "season",
    # Lag features
    "lag_1h", "lag_2h", "lag_3h", "lag_24h", "lag_168h",
    # Rolling means
    "roll_mean_3h", "roll_mean_24h",
    # Road static
    "road_subclass_enc", "longitude", "latitude",
    # POI proximity (NEW)
    "dist_station_km", "dist_stadium_km", "dist_hospital_km",
    "dist_university_km", "dist_mall_km", "dist_bus_station_km",
    "near_major_poi",
]
target_col = "target"

print(f"Total features: {len(feature_cols)}")

Total features: 27


In [29]:
# Update the scaler to include POI features
# Refit on training data with the new columns
stats = con.sql(f"""
    SELECT {", ".join([f"MIN({c}) AS min_{c}, MAX({c}) AS max_{c}"
                       for c in feature_cols + [target_col]])}
    FROM read_parquet('/home/iai-labor/traffic_project/traffic_features_v1.parquet')
    WHERE split = 'train'
""").df()

import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

X_min = np.array([stats[f"min_{c}"].values[0] for c in feature_cols])
X_max = np.array([stats[f"max_{c}"].values[0] for c in feature_cols])

scaler_X = MinMaxScaler()
scaler_X.fit(np.vstack([X_min, X_max]))
scaler_X.data_min_ = X_min
scaler_X.data_max_ = X_max
scaler_X.scale_ = 1.0 / (X_max - X_min + 1e-8)
scaler_X.min_ = -X_min * scaler_X.scale_
scaler_X.data_range_ = X_max - X_min
scaler_X.n_features_in_ = len(feature_cols)

joblib.dump(scaler_X, "/home/iai-labor/traffic_project/scaler_X.pkl")

# Also save target scaler (y)
y_min_val = stats[f"min_target"].values[0]
y_max_val = stats[f"max_target"].values[0]

scaler_y = MinMaxScaler()
scaler_y.fit(np.array([[y_min_val], [y_max_val]]))
scaler_y.data_min_ = np.array([y_min_val])
scaler_y.data_max_ = np.array([y_max_val])
scaler_y.scale_ = np.array([1.0 / (y_max_val - y_min_val + 1e-8)])
scaler_y.min_ = np.array([-y_min_val * scaler_y.scale_[0]])
scaler_y.data_range_ = np.array([y_max_val - y_min_val])
scaler_y.n_features_in_ = 1

joblib.dump(scaler_y, "/home/iai-labor/traffic_project/scaler_y.pkl")
print(f"Target range (train): {y_min_val:.1f} to {y_max_val:.1f}")

print(f"Scaler updated with {len(feature_cols)} features")
print(f"POI station range: {stats['min_dist_station_km'].values[0]:.3f} "
      f"to {stats['max_dist_station_km'].values[0]:.2f} km")

Target range (train): 0.0 to 90910.0
Scaler updated with 27 features
POI station range: 0.002 to 3.74 km


In [30]:
# ── is_main_artery feature ─────────────────────────────────────────────────
# Identifies sensors on physically important roads regardless of admin class
# OSM highway hierarchy: motorway/trunk = ring roads and expressways
#                        primary/secondary = major urban through-roads
#                        motorway_link/trunk_link = on/off ramps

import pandas as pd

sensor_road_attrs = pd.read_parquet(
    "/home/iai-labor/traffic_project/sensor_road_attributes.parquet"
)

# Convert lanes to numeric in case it's stored as string
sensor_road_attrs["lanes_numeric"] = pd.to_numeric(
    sensor_road_attrs["lanes"], errors="coerce"
)

ARTERY_HIGHWAY_TYPES = [
    "motorway", "motorway_link",   # M30, C30, ramp connectors
    "trunk", "trunk_link",          # major expressways
    "primary", "secondary",         # main urban arterials
    "primary_link", "secondary_link"
]

artery_ids = set(
    sensor_road_attrs[
        sensor_road_attrs["highway"].isin(ARTERY_HIGHWAY_TYPES)
    ]["SENSOR_ID"].unique()
)

print(f"Main artery sensors: {len(artery_ids)} of {len(sensor_road_attrs)}")
print()

# Breakdown by road subclass to verify it makes intuitive sense
check = sensor_road_attrs.copy()
check["is_main_artery"] = check["SENSOR_ID"].isin(artery_ids).astype(int)
print(check.groupby(["ELEMENT_TYPE", "is_main_artery"])["SENSOR_ID"].count().unstack())
print()

# Breakdown by highway type within artery sensors
artery_rows = sensor_road_attrs[sensor_road_attrs["SENSOR_ID"].isin(artery_ids)]
print("Highway types within artery sensors:")
print(artery_rows["highway"].value_counts())

Main artery sensors: 2130 of 4845

is_main_artery     0     1
ELEMENT_TYPE              
C30               12   132
M30                7   293
URB             2696  1705

Highway types within artery sensors:
highway
primary           822
secondary         715
motorway_link     278
motorway          212
trunk              59
primary_link       28
secondary_link     15
trunk_link          1
Name: count, dtype: int64


In [31]:
# Save artery lookup and join into features file
artery_df = pd.DataFrame({
    "SENSOR_ID": list(artery_ids),
    "is_main_artery": 1
})

# Save as a small reference file
artery_df.to_parquet(
    "/home/iai-labor/traffic_project/sensor_artery_flags.parquet",
    index=False
)

# Add is_main_artery to traffic_features_v2.parquet via DuckDB
con.sql(f"""
    COPY (
        SELECT f.*,
               COALESCE(a.is_main_artery, 0) AS is_main_artery
        FROM read_parquet('{features_path}') f
        LEFT JOIN read_parquet(
            '/home/iai-labor/traffic_project/sensor_artery_flags.parquet'
        ) a ON f.SENSOR_ID = a.SENSOR_ID
    ) TO '/home/iai-labor/traffic_project/traffic_features_v2.parquet'
    (FORMAT PARQUET, COMPRESSION 'snappy')
""")

print("Saved traffic_features_v2.parquet")

# Quick verify
con.sql("""
    SELECT is_main_artery, COUNT(*) AS n_rows,
           COUNT(DISTINCT SENSOR_ID) AS n_sensors
    FROM read_parquet('/home/iai-labor/traffic_project/traffic_features_v2.parquet')
    GROUP BY is_main_artery
""").show()

Saved traffic_features_v2.parquet
┌────────────────┬──────────┬───────────┐
│ is_main_artery │  n_rows  │ n_sensors │
│     int64      │  int64   │   int64   │
├────────────────┼──────────┼───────────┤
│              1 │ 61430584 │      2126 │
│              0 │ 78556424 │      2760 │
└────────────────┴──────────┴───────────┘



In [45]:
# Final feature column list — 28 features (27 + is_main_artery)
feature_cols = [
    # Temporal cyclical
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    # Calendar flags
    "is_weekend", "is_holiday", "is_rush_hour", "season",
    # Lag features
    "lag_1h", "lag_2h", "lag_3h", "lag_24h", "lag_168h",
    # Rolling means
    "roll_mean_3h", "roll_mean_24h",
    # Road static
    "road_subclass_enc", "longitude", "latitude",
    # POI proximity
    "dist_station_km", "dist_stadium_km", "dist_hospital_km",
    "dist_university_km", "dist_mall_km", "dist_bus_station_km",
    "near_major_poi",
    # Main artery flag (NEW)
    "is_main_artery",
]
target_col = "target"

print(f"Total features: {len(feature_cols)}") 

Total features: 28
Scalers saved: 28 features, target 0–90910


In [53]:
# Compute per-sensor 99.9th percentile from TRAINING data only
# Values above this threshold for a given sensor are almost certainly
# hardware faults, not real traffic — a sensor that normally sees 300
# vehicles/hour cannot legitimately report 90,000

sensor_caps = con.sql(f"""
    SELECT
        SENSOR_ID,
        PERCENTILE_CONT(0.999) WITHIN GROUP (ORDER BY target) AS cap_999,
        AVG(target) AS avg_intensity,
        MAX(target) AS raw_max
    FROM read_parquet('{features_path}')
    WHERE split = 'train' AND target IS NOT NULL AND target > 0
    GROUP BY SENSOR_ID
""").df()

# A sensor near a train station might legitimately have cap_999 = 5000
# A quiet residential sensor might have cap_999 = 400
# Sensor 7023 (faulty) would have cap_999 around 85000+ — still faulty

print(sensor_caps.nlargest(10, "cap_999")[["SENSOR_ID","cap_999","avg_intensity","raw_max"]])
print()
print(f"Sensor 7023 stats:")
print(sensor_caps[sensor_caps["SENSOR_ID"] == 7023])

      SENSOR_ID       cap_999  avg_intensity       raw_max
4608       5465  56863.455500     529.659449  90910.000000
613       10674  24553.002500     260.204072  42949.666667
3414       6070  19168.688750     429.464914  25120.750000
3073       5470  18057.194352     387.117559  66255.000000
3011       5767  12416.001250     749.370238  89515.500000
4564       4059  12178.510750    1170.454298  26872.750000
1714       6920  11037.130000     685.891251  20774.000000
1823       6069  10890.296500     430.572720  15719.750000
2356      10298  10575.413500     572.803140  13118.000000
3653       6940   9495.524750     871.419844  32168.750000

Sensor 7023 stats:
      SENSOR_ID   cap_999  avg_intensity  raw_max
1858       7023  1237.708     148.169893  4756.75


In [55]:
# Cap outliers at 10x each sensor's own median — not a global threshold
# This preserves genuine high-traffic sensors (7023 near station stays)
# while removing hardware faults (5465 spiking to 165x its own average)
# Reference: Vélez-Serrano (2021) — "clipped at mean plus 2 standard deviations"

sensor_stats = con.sql(f"""
    SELECT
        SENSOR_ID,
        MEDIAN(target) AS median_intensity,
        AVG(target) AS avg_intensity,
        STDDEV(target) AS std_intensity,
        -- Cap at mean + 3*std (classic outlier rule, per sensor)
        AVG(target) + 3 * STDDEV(target) AS cap_3std
    FROM read_parquet('{features_path}')
    WHERE split = 'train'
      AND target IS NOT NULL
      AND target > 0
    GROUP BY SENSOR_ID
""").df()

# Save this for use during model training (to clip values before scaling)
sensor_stats.to_parquet(
    "/home/iai-labor/traffic_project/sensor_outlier_caps.parquet",
    index=False
)

# Preview: what does this give sensor 7023 vs sensor 5465?
check_sensors = [7023, 5465, 10674, 5767]
print("Per-sensor caps (mean + 3*std):")
print(sensor_stats[sensor_stats["SENSOR_ID"].isin(check_sensors)][
    ["SENSOR_ID", "avg_intensity", "std_intensity", "cap_3std"]
].round(1))

Per-sensor caps (mean + 3*std):
      SENSOR_ID  avg_intensity  std_intensity  cap_3std
159       10674          260.2         1335.9    4267.8
1218       5767          749.4         1182.9    4298.0
2011       7023          148.2          113.9     489.8
4495       5465          529.7         2640.3    8450.5
